# Analyse toevoegen handleiding instructies
Dit notebook ondersteunt het verkennen, analyseren en valideren van de handleiding en instructies functionaliteit binnen AskDelphi.  
De module biedt helpermethoden voor het toevoegen, beheren en inspecteren van handleiding en instructie relaties die aan AskDelphi‑topics gekoppeld zijn.  
Het doel van dit notebook is om:

- inzicht te krijgen in de datastructuren en API‑interacties rondom handleidingen en instructies
- voorbeelddata te analyseren en patronen te ontdekken
- helpermethoden te testen en gedrag te evalueren
- experimentele of onderzoeksgerichte analyses uit te voeren ter verbetering van de contentstructuur in AskDelphi

Dit notebook vormt daarmee een flexibel startpunt voor verdere exploratie of debugging van de Handleidingen en instructies zaken.

### Functie toevoegen aan relation.py

In [1]:
def get_relationTypeId_by_relationTypeName(topic_id_action : str, topic_version_id_action: str, relationTypeName: str) -> str:
    # Welke relatie types zijn er?
    endpoint = f"/v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topic_id_action}/topicVersion/{topic_version_id_action}/allowedrelations"
    result = client._request(
        "GET", 
        endpoint,
        json_data={}
    )

    # relationTypeId Handleidingen en instructies
    relationTypeId = ""
    for item in result["topicAllowedRelations"]:
        # print(item)
        if item['relationTypeName'] == "Handleidingen en instructies":
            print(f"{item['relationTypeName']} => relationTypeId {item["relationTypeId"]}")
            relationTypeId = item["relationTypeId"]
            break
    
    return relationTypeId

### Connectie met AskDelphi opzetten

In [2]:
import uuid
from ask_delphi_api.authentication import AskDelphiClient
client = AskDelphiClient()
client.authenticate()   # pakt automatisch portal code uit .env

AskDelphi Client loaded.
Loaded cached tokens.
AUTHENTICATION STARTED
Trying cached tokens...
Editing API token status: 200
Editing API token acquired.
SUCCESS using cached tokens!


True

In [3]:
from ask_delphi_api.project import Project
from ask_delphi_api.topictools import TopicTools
from ask_delphi_api.relation import Relation
from ask_delphi_api.workflow import Workflow
workflow = Workflow(client)
project = Project(client)
topic = TopicTools(client, project)
relation = Relation(client)

### Toevoegen handleiding

In [ ]:
# Checkout Action topic
# topic_id = "f61f3645-72cc-444b-b504-346699950682" #Open postbak in WAB
topic_id = "63d41500-76a0-48e7-bab9-274cd0cbf144" #selecteer een mooie zomerdag zzz omgeving
result = topic.checkout(topic_id)
topic_version_id = result['topicVersionId']
print(f"topic_id : {topic_id}")
print(f"topic_version_id : {topic_version_id}")

ValueError: tenant_id is not set. Provide cms_url or explicit IDs to the constructor.

In [ ]:
# Welke relatie types zijn er?
endpoint = f"/v2/tenant/{{tenantId}}/project/{{projectId}}/acl/{{aclEntryId}}/topic/{topic_id}/topicVersion/{topic_version_id}/allowedrelations"
result = client._request(
    "GET", 
    endpoint,
    json_data={}
)

# relationTypeId verschilt per topicType
for item in result["topicAllowedRelations"]:
    if item['pyramidLevelName'] == "Handleidingen en Instructies":
        # print(item['topicTypeName'], item["relationTypeId"])
        print(item)

In [ ]:
# Toevoegen Externe URL aan action
topic_id_externe_url = str(uuid.uuid4())      
topicTitle = "Externe URL download ab 3"      
topicTypeId = project.get_topic_type_id("External URL")    
parentTopicId = topic_id
parentTopicVersionId = topic_version_id
parentTopicRelationTypeId = get_relationTypeId_by_relationTypeName(topic_id, topic_version_id, "Handleidingen en instructies")
print(f"parentTopicRelationTypeId : {parentTopicRelationTypeId}")

relation.add_topic_with_relation(client, topic_id_externe_url, topicTitle, topicTypeId, parentTopicId, parentTopicRelationTypeId, parentTopicVersionId)

In [ ]:
# Checkin taak topic
topic.checkin(topic_id)
print(f"Checked in task topic : {topic_id}")